<a href="https://colab.research.google.com/github/nileshpatil1206070/KTI_demo_student-feedback-nlp-pipeline/blob/main/KTI_projects_3rd_point_student_feedback.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# i will install all necessary libraries

!pip install -q sentence-transformers openpyxl pandas numpy scikit-learn nltk

import os
import pandas as pd
import numpy as np
import nltk
from google.colab import drive
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer

# Download the VADER lexicon required for Sentiment Intensity Analysis
nltk.download('vader_lexicon', quiet=True)

True

In [ ]:
# =====================================================================
# STEP 2: MOUNT GOOGLE DRIVE & LOAD FILE
# =====================================================================

from google.colab import drive
drive.mount('/content/drive')

# UPDATE THIS PATH to point exactly where your file lives in your Google Drive
# For example: '/content/drive/MyDrive/Student Feedback form.xlsx'


Mounted at /content/drive


In [ ]:
file_path="/content/drive/MyDrive/KTI_Dataset/Student Feedback form.xlsx"

In [ ]:
if not os.path.exists(file_path):
  raise FileNotFoundError(f"file is not present at {file_path}. Please check your google drive path")

In [ ]:
xls = pd.ExcelFile(file_path)
raw_feedback =[]

for sheet in xls.sheet_names:
  df_sheet=pd.read_excel(file_path, sheet_name=sheet)
  texts=df_sheet['Feedback'].dropna().astype(str)
  raw_feedback.extend(texts)

df=pd.DataFrame({'Feedback':raw_feedback})

In [ ]:
analyzer = SentimentIntensityAnalyzer()
df['Sentiment_Score']=df['Feedback'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
print(df.head())
df['Sentiment']=df['Sentiment_Score'].apply(lambda s: 'Positive' if s > 0.1 else ('Negative' if s < -0.1 else 'Neutral'))
print(df.head())

                                            Feedback  Sentiment_Score
0  Fairly simple and straight forward platform to...           0.2263
1  however I did have to out source from the refe...           0.0000
2            provided to locate some of the answers.           0.0000
3                                               Good           0.4404
4  Wrong Answers should be given chance to be cor...          -0.3527
                                            Feedback  Sentiment_Score  \
0  Fairly simple and straight forward platform to...           0.2263   
1  however I did have to out source from the refe...           0.0000   
2            provided to locate some of the answers.           0.0000   
3                                               Good           0.4404   
4  Wrong Answers should be given chance to be cor...          -0.3527   

  Sentiment  
0  Positive  
1   Neutral  
2   Neutral  
3  Positive  
4  Negative  


In [ ]:


# Isolate only negative feedback for improvement analysis
negative_feedback_df = df[df['Sentiment'] == 'Negative'].copy()
negative_list = negative_feedback_df['Feedback'].tolist()

print(f"Dynamically Isolated Purely Negative Feedbacks: {len(negative_list)}\n")

# =====================================================================
# APPROACH 1: LITERAL COUNT VECTORIZER (Frequency & Feature Extraction)
# =====================================================================
print("="*60)
print("APPROACH 1: N-GRAM COUNT VECTORIZER PATTERNS")
print("="*60)

if len(negative_list) > 0:
    count_vectorizer = CountVectorizer(stop_words='english', ngram_range=(1, 3))
    print(count_vectorizer)
    X_counts = count_vectorizer.fit_transform(negative_list)

    # Sum phrase occurrences across the text corpus
    common_phrases = pd.Series(
        X_counts.sum(axis=0).A1,
        index=count_vectorizer.get_feature_names_out()
    ).sort_values(ascending=False)

    print("Top 5 exact recurring terminology markers surfacing in complaints:")
    print(common_phrases.head(5))
else:
    print("No negative feedback items found to analyze.")
print("\n")

Dynamically Isolated Purely Negative Feedbacks: 213

APPROACH 1: N-GRAM COUNT VECTORIZER PATTERNS
CountVectorizer(ngram_range=(1, 3), stop_words='english')
Top 5 exact recurring terminology markers surfacing in complaints:
questions      68
hard           36
difficult      27
question       20
information    19
dtype: int64




In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# Isolate only negative feedback for improvement analysis
negative_feedback_df = df[df['Sentiment'] == 'Negative'].copy()
negative_list = negative_feedback_df['Feedback'].tolist()

print(f"Dynamically Isolated Purely Negative Feedbacks: {len(negative_list)}\n")

# =====================================================================
# APPROACH 1: LITERAL COUNT VECTORIZER (Frequency & Feature Extraction)
# =====================================================================
print("="*60)
print("APPROACH 1: N-GRAM COUNT VECTORIZER PATTERNS")
print("="*60)

if len(negative_list) > 0:
    # -------------------------------------------------------------
    # STEP 1: INITIALIZE THE VECTORIZER
    # -------------------------------------------------------------
    print("[STEP 1] Initializing CountVectorizer...")
    count_vectorizer = CountVectorizer(stop_words='english', ngram_range=(1, 3))
    print(" -> Vectorizer is ready.\n")

    # -------------------------------------------------------------
    # STEP 2: FIT & TRANSFORM (Learning Vocabulary & Creating Matrix)
    # -------------------------------------------------------------
    print("[STEP 2] Fitting & transforming raw text into a mathematical matrix...")
    X_counts = count_vectorizer.fit_transform(negative_list)

    # Get vocabulary lists
    vocabulary = count_vectorizer.get_feature_names_out()
    print(f" -> Matrix shape: {X_counts.shape} ({X_counts.shape[0]} reviews mapped across {X_counts.shape[1]} unique vocabulary combinations)")

    # Visualizing how the first row looks as mathematical vector coordinates
    print(f" -> Text of first review: \"{negative_list[0]}\"")
    print(f" -> Row #0 converted to vector coordinates: {X_counts[0].toarray().flatten()}\n")

    # -------------------------------------------------------------
    # EXPORTING THE ENTIRE MATRIX GRID TO CSV
    # -------------------------------------------------------------
    print("[STEP 2.5] Saving the FULL mathematical grid (0s and 1s matrix) to a CSV file...")

    # Rows represent individual student complaints, columns are the vocabulary terms
    df_full_matrix = pd.DataFrame(X_counts.toarray(), columns=vocabulary)

    # We insert the raw student comments as the very first column so you can see which text matches which row of 0s and 1s!
    df_full_matrix.insert(0, 'Original_Student_Feedback', negative_list)

    # Save file
    df_full_matrix.to_csv('student_feedback_matrix.csv', index=False)
    print(" -> SUCCESS: Saved complete matrix to file: 'student_feedback_matrix.csv'")
    print(f"    (Open this file in Excel to inspect all {X_counts.shape[0]} rows and {X_counts.shape[1]} phrase columns!)\n")

    # -------------------------------------------------------------
    # STEP 3: SUM PHRASE OCCURRENCES
    # -------------------------------------------------------------
    print("[STEP 3] Summing occurrence counts down each column...")
    raw_sums = X_counts.sum(axis=0) # Sum each column across all rows
    flat_sums = raw_sums.A1         # Flatten matrix down to a 1D array of total counts

    # Print the raw list of counts (we print a sample but we will save the FULL list below)
    print(f" -> Total counts list preview: {list(flat_sums[:15])} ...")
    print(" -> Summing complete.\n")

    # -------------------------------------------------------------
    # STEP 4: PAIR SUMS BACK WITH WORD LABELS, SORT, & EXPORT TO CSV
    # -------------------------------------------------------------
    print("[STEP 4] Mapping total numbers back to their English phrase labels and sorting in descending order...")
    common_phrases_series = pd.Series(
        flat_sums,
        index=vocabulary
    ).sort_values(ascending=False)

    print("[STEP 4.5] Saving the FULL sorted frequency list to a CSV file...")
    # Convert series to a clean DataFrame for export
    df_sorted_frequencies = common_phrases_series.reset_index()
    df_sorted_frequencies.columns = ['Phrase', 'Total_Count_Frequency']

    # Save file
    df_sorted_frequencies.to_csv('full_sorted_phrases_frequency.csv', index=False)
    print(" -> SUCCESS: Saved full sorted list to file: 'full_sorted_phrases_frequency.csv'")
    print(f"    (Open this file in Excel to see all {len(df_sorted_frequencies)} words/phrases sorted by highest count!)\n")

    # -------------------------------------------------------------
    # STEP 5: FINAL SUMMARY
    # -------------------------------------------------------------
    print("[STEP 5] Printing final executive overview:")
    print("-" * 50)
    print("Top 10 exact recurring terminology markers surfacing in complaints:")
    print(common_phrases_series.head(10))
    print("-" * 50)

else:
    print("No negative feedback items found to analyze.")
print("\n")

Dynamically Isolated Purely Negative Feedbacks: 213

APPROACH 1: N-GRAM COUNT VECTORIZER PATTERNS
[STEP 1] Initializing CountVectorizer...
 -> Vectorizer is ready.

[STEP 2] Fitting & transforming raw text into a mathematical matrix...
 -> Matrix shape: (213, 2332) (213 reviews mapped across 2332 unique vocabulary combinations)
 -> Text of first review: "Wrong Answers should be given chance to be corrected accordingly after question? To recall the right and proper one?"
 -> Row #0 converted to vector coordinates: [0 0 0 ... 0 0 0]

[STEP 2.5] Saving the FULL mathematical grid (0s and 1s matrix) to a CSV file...
 -> SUCCESS: Saved complete matrix to file: 'student_feedback_matrix.csv'
    (Open this file in Excel to inspect all 213 rows and 2332 phrase columns!)

[STEP 3] Summing occurrence counts down each column...
 -> Total counts list preview: [np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(1

In [ ]:



# =====================================================================
# APPROACH 2: SEMANTIC THEME CLUSTERING (Sentence Transformers)
# =====================================================================
print("="*60)
print("APPROACH 2: SENTENCE-LEVEL SEMANTIC CONTEXT ANALYSIS")
print("="*60)

if len(negative_list) > 0:
    # Load a local lightweight sentence embedded model (Runs locally on Colab free CPU/GPU)
    semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

    # Transform full sentence texts into numeric vector semantic matrices
    embeddings = semantic_model.encode(negative_list, show_progress_bar=False)

    # Partition meanings into 3 macro operational friction buckets
    num_clusters = min(3, len(negative_list))
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    negative_feedback_df['Cluster_ID'] = kmeans.fit_predict(embeddings)

    # Map contextual variations together and extract suggestions
    for cluster in range(num_clusters):
        cluster_subset = negative_feedback_df[negative_feedback_df['Cluster_ID'] == cluster]
        cluster_embeddings = embeddings[negative_feedback_df['Cluster_ID'] == cluster]

        # Identify the exact sentence closest to the cluster center to act as the Theme Anchor
        centroid = kmeans.cluster_centers_[cluster]
        distances = np.linalg.norm(cluster_embeddings - centroid, axis=1)
        representative_phrase = cluster_subset.iloc[distances.argmin()]['Feedback']

        print("representative_phrase:",representative_phrase)
        print('\n===========================================')
        print(f"\n[Semantic Category Theme #{cluster + 1}] (Volume: {len(cluster_subset)} entries)")
        print(f" -> Root Meaning Core: \"{representative_phrase}\"")
        print(" -> Real Student Variance Samples grouped here:")

        # Display unique phrasings dropped into the exact same category bucket
        samples = cluster_subset['Feedback'].drop_duplicates().head(2).tolist()
        for s in samples:
            print(f"    * \"{s}\"")

        # GENERATE DATA-DRIVEN IMPROVEMENT SUGGESTIONS
        print(" -> Actionable System Improvement Suggestion:")
        if "navigate" in representative_phrase.lower() or "site" in representative_phrase.lower():
            print("    >> Overhaul user onboarding. Add a short guide on portal usage before launching assessment frames.")
        elif "question" in representative_phrase.lower() or "test" in representative_phrase.lower() or "hard" in representative_phrase.lower():
            print("    >> Audit validation question layouts. Review ambiguous parsing logic and offer instant post-quiz correction context.")
        else:
            print("    >> Optimize system responsiveness, session length limits, or operational trainer-to-student support windows.")
else:
    print("No negative feedback items found to generate semantic clusters.")

APPROACH 2: SENTENCE-LEVEL SEMANTIC CONTEXT ANALYSIS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

representative_phrase: very poor wording of questions


[Semantic Category Theme #1] (Volume: 72 entries)
 -> Root Meaning Core: "very poor wording of questions"
 -> Real Student Variance Samples grouped here:
    * "Wrong Answers should be given chance to be corrected accordingly after question? To recall the right and proper one?"
    * "some tricky question worded a bit confusingly"
 -> Actionable System Improvement Suggestion:
    >> Audit validation question layouts. Review ambiguous parsing logic and offer instant post-quiz correction context.
representative_phrase: Very difficult to navigate Couldn’t find any course to review just went straight to the assessment


[Semantic Category Theme #2] (Volume: 100 entries)
 -> Root Meaning Core: "Very difficult to navigate Couldn’t find any course to review just went straight to the assessment"
 -> Real Student Variance Samples grouped here:
    * "Good information but difficult to navigate site."
    * "Hard"
 -> Actionable System Impro

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

print("="*60)
print("APPROACH 2: TARGETED OPERATIONAL SEMANTIC CLUSTERING")
print("="*60)

if len(negative_list) > 0:
    # 1. Initialize local open-source transformer model
    semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

    # 2. DEFINE YOUR EXACT TARGET GROUND CATEGORIES
    # The model will use these definitions to judge the semantic intent of student text
    target_categories = {
        "Navigation & Portal UI": {
            "definition_anchor": "portal navigation website interface site layout difficult to navigate find links pages dashboard layout",
            "manager_action": "[ALERT: UI/UX Friction] Manager to decide if portal interface updates or a student onboarding guide is required."
        },
        "Assessment & Question Design": {
            "definition_anchor": "quiz question test exam answers wording confusing hard choices vague options grading grading layout failure",
            "manager_action": "[ALERT: Curriculum Risk] Manager to audit high-failure quiz modules and review ambiguous question text layouts."
        },
        "System Speed & Performance": {
            "definition_anchor": "slow timing timeout server lag loading freeze crash timing out system responsiveness speed delay loading pages",
            "manager_action": "[ALERT: Technical Infrastructure Risk] Manager to audit hosting server responsiveness and adjust session timeout length caps."
        }
    }

    # 3. Create Vector Embeddings for our target category definitions
    category_names = list(target_categories.keys())
    definition_phrases = [target_categories[cat]["definition_anchor"] for cat in category_names]
    category_vectors = semantic_model.encode(definition_phrases, show_progress_bar=False)

    # 4. Create Vector Embeddings for all the raw negative student comments
    student_vectors = semantic_model.encode(negative_list, show_progress_bar=False)

    # 5. Calculate Similarity Matrix between every student vector and our target categories
    # Cosine similarity yields a score between -1 and 1 representing closeness of structural meaning
    similarity_matrix = cosine_similarity(student_vectors, category_vectors)

    # 6. Assign each student complaint to its highest-scoring target category
    assigned_category_indices = np.argmax(similarity_matrix, axis=1)

    # Map back to our dataframe for clean grouping
    negative_feedback_df['Assigned_Category'] = [category_names[i] for i in assigned_category_indices]
    negative_feedback_df['Max_Similarity_Score'] = np.max(similarity_matrix, axis=1)

    # 7. Print the cleanly organized, targeted executive summary
    for cat_name in category_names:
        cluster_subset = negative_feedback_df[negative_feedback_df['Assigned_Category'] == cat_name]

        print(f"\n=======================================================")
        print(f"[Semantic Category: {cat_name}] (Volume: {len(cluster_subset)} entries)")
        print(f"=======================================================")

        if len(cluster_subset) > 0:
            # Find the true core issue: the sentence within this group that matched our target category definition the closest
            top_match_idx = cluster_subset['Max_Similarity_Score'].idxmax()
            representative_phrase = cluster_subset.loc[top_match_idx, 'Feedback']

            print(f" -> Core System Issue Highlighted: \"{representative_phrase}\"")
            print(" -> Distinct Student Text Variants mapped directly to this ground:")

            # Print unique student samples grouped here
            samples = cluster_subset['Feedback'].drop_duplicates().head(2).tolist()
            for s in samples:
                print(f"    * \"{s}\"")

            # Output the explicit strategic directive linked directly to this precise group
            print(f" -> Actionable Operational Brief for Manager Decision:\n    >> {target_categories[cat_name]['manager_action']}")
        else:
            print(" -> No student feedback items mapped to this category ground during this analysis window.")

else:
    print("No negative feedback items found to categorize.")

APPROACH 2: TARGETED OPERATIONAL SEMANTIC CLUSTERING


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


[Semantic Category: Navigation & Portal UI] (Volume: 22 entries)
 -> Core System Issue Highlighted: "Your portal is hard to navigate"
 -> Distinct Student Text Variants mapped directly to this ground:
    * "Good information but difficult to navigate site."
    * "was hard for me to navigate as i dont do so much online learning"
 -> Actionable Operational Brief for Manager Decision:
    >> [ALERT: UI/UX Friction] Manager to decide if portal interface updates or a student onboarding guide is required.

[Semantic Category: Assessment & Question Design] (Volume: 168 entries)
 -> Core System Issue Highlighted: "Poor English and poorly worded questions in the assessment. Made it confusing and some questions don't have the correct answer as an option"
 -> Distinct Student Text Variants mapped directly to this ground:
    * "Wrong Answers should be given chance to be corrected accordingly after question? To recall the right and proper one?"
    * "Hard"
 -> Actionable Operational Brief for M

**Best working approach**

This pure machine learning pipeline achieves unsupervised theme discovery by



1.first converting all raw, negative text strings into high-dimensional vector embeddings using an open-source `SentenceTransformer`, which maps sentences into a mathematical landscape based strictly on abstract contextual meaning rather than literal characters. The unsupervised `K-Means` algorithm then scans this spatial landscape to automatically identify natural data clouds and establish independent geometric center points, known as centroids, for each cluster without any pre-configured category labels. By measuring the straight-line Euclidean distance from these centroids to every individual vector point in their respective clouds, the code uses `distances.argmin()` to isolate the specific verbatim comment possessing the absolute minimum distance to serve as the cluster’s authentic Root Meaning Core.

2.The defining importance of this spatial math approach over the previous hardcoded keyword architecture lies in its absolute elimination of human assumption, algorithmic bias, and rigid substring matching. Instead of lazily dropping unexpected student complaints into a generic fallback bucket or misclassifying critical course friction because a student used an unmapped synonym like "brutal" instead of "hard," the machine objectively lets the geometric proximity of the data dictate the categories, surfacing the pure consensus voice of the dataset to construct an unbiased, highly accurate operational brief for the training manager's final strategic review.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
#from sentence_transformers import Transformer

print("="*75)
print("APPROACH 2: PURE UNSUPERVISED ML TOPIC DISCOVERY & CENTROID ANALYSIS")
print("="*75)

if len(negative_list) > 0:
    # 1. Initialize local, open-source transformer model to create sentence embeddings
    semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = semantic_model.encode(negative_list, show_progress_bar=False)

    # 2. Let ML dynamically discover 3 natural clusters on its own ground
    num_clusters = min(3, len(negative_list))
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    negative_feedback_df['Cluster_ID'] = kmeans.fit_predict(embeddings)

    # 3. Iterate through each machine-learned cluster to find its spatial center
    for cluster in range(num_clusters):
        cluster_subset = negative_feedback_df[negative_feedback_df['Cluster_ID'] == cluster]
        cluster_embeddings = embeddings[negative_feedback_df['Cluster_ID'] == cluster]

        # Calculate the mathematical center point (Centroid) of this specific group
        centroid = kmeans.cluster_centers_[cluster]

        # Calculate the distance from the cluster center to all individual points in this cluster
        distances = np.linalg.norm(cluster_embeddings - centroid, axis=1)

        # The sentence with the ABSOLUTE MINIMUM DISTANCE becomes the core theme defined by the ML
        representative_phrase = cluster_subset.iloc[distances.argmin()]['Feedback']

        print(f"\n[Machine-Learned Category Cluster #{cluster + 1}] (Volume: {len(cluster_subset)} student entries)")
        print(f" -> Automatically Discovered Core Issue: \"{representative_phrase}\"")
        print(" -> Sample Verbatim Variations grouped into this mathematical space:")

        # Show real variation samples that the ML grouped into this neighborhood
        samples = cluster_subset['Feedback'].drop_duplicates().head(2).tolist()
        for s in samples:
            print(f"    * \"{s}\"")

        # 4. Present structured data directly to the Training Manager for the final decision
        print(" -> Actionable Operational Brief for Management Review:")
        print(f"    [DIRECTIVE] This cluster has formed entirely around the core problem: '{representative_phrase}'.")
        print(f"    [ACTION REQUIRED] Training Manager to audit the {len(cluster_subset)} logged entries in this bucket")
        print(f"    and engineer a targeted operational, system, or curriculum intervention based on this pattern.")
        print("-"*75)
else:
    print("No negative feedback items found to generate machine-learned clusters.")

APPROACH 2: PURE UNSUPERVISED ML TOPIC DISCOVERY & CENTROID ANALYSIS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


[Machine-Learned Category Cluster #1] (Volume: 72 student entries)
 -> Automatically Discovered Core Issue: "very poor wording of questions"
 -> Sample Verbatim Variations grouped into this mathematical space:
    * "Wrong Answers should be given chance to be corrected accordingly after question? To recall the right and proper one?"
    * "some tricky question worded a bit confusingly"
 -> Actionable Operational Brief for Management Review:
    [DIRECTIVE] This cluster has formed entirely around the core problem: 'very poor wording of questions'.
    [ACTION REQUIRED] Training Manager to audit the 72 logged entries in this bucket
    and engineer a targeted operational, system, or curriculum intervention based on this pattern.
---------------------------------------------------------------------------

[Machine-Learned Category Cluster #2] (Volume: 100 student entries)
 -> Automatically Discovered Core Issue: "Very difficult to navigate Couldn’t find any course to review just went str